# CIFAR-100 learning-rate sanity check --- ResNet-34

Does the CIFAR-10 protocol transfer to a 100-class task before we spend a
night on the full grid?

The matched sweep runs I.P. at SGD 0.01 and BaCP at 0.1, a split measured on
CIFAR-10. A 100-way cross-entropy has a different loss scale than a 10-way
one, so that split is an assumption here, not a measurement.

It is the assumption that already failed once. MobileNetV2 inherited the
ResNet recipe untuned and its I.P. arm collapsed to chance at 0.99 under
0.01 while reaching 78.63 under 0.1 --- a difference that would have been
reported as a seventy-point win for BaCP had it gone unchecked. That cost
two GPU-hours to find. Finding the same thing here costs about thirty
minutes; missing it costs the twelve-hour grid.

## What this runs

Three cells, and only two of them are diagnostic:

1. `dense` --- ResNet-34 / CIFAR-100 / seed 1. **Not throwaway**: every sparse
   cell of the real grid resolves its checkpoint from the dense run of the
   same seed, so this is work the full sweep needs regardless. It is skipped
   automatically when the grid runs later.
2. `prune` at 0.95 under **LR 0.01** --- the protocol rate.
3. `prune` at 0.95 under **LR 0.1** --- the alternative.

Only `learning_rate` differs between 2 and 3; the variant suffix keeps the
0.1 record out of the main namespace.

## How to read it

- **0.01 wins, or they are within the ~0.3-point noise floor** --- the
  protocol transfers. Run the full grid as specified.
- **0.1 wins clearly** --- the CIFAR-10 split does not transfer, and the grid
  must be re-planned before it runs, exactly as MobileNetV2 had to be.

0.95 is the right sparsity to probe on: it is the least damaged cell, so any
gap here is about optimisation rather than capacity, and it is the cheapest
sparse cell to run.


In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Preflight

Halts before any GPU time if the model is not actually pretrained.

In [ ]:
MODEL, DATASET, NCLS, SEED, GPU = 'resnet34', 'cifar100', 100, 1, 0
nb.fetch_imagenet_weights(MODEL)
nb.preflight(MODEL, num_classes=NCLS)

## Dense baseline

Needed by the full grid too, so this is not probe-only cost.

In [ ]:
dense = nb.make_cell(MODEL, 'dense', seed=SEED,
                     dataset_name=DATASET, num_classes=NCLS)
print(dense['key'])
nb.run(dense, gpu=GPU)

## The probe

Two I.P. cells at 0.95, identical but for the learning rate.

In [ ]:
a = nb.make_cell(MODEL, 'prune', seed=SEED, pruner='magnitude', sparsity=0.95,
                 dataset_name=DATASET, num_classes=NCLS)                 # lr 0.01
b = nb.make_cell(MODEL, 'prune', seed=SEED, pruner='magnitude', sparsity=0.95,
                 dataset_name=DATASET, num_classes=NCLS,
                 variant='lr0.1', learning_rate=0.1)

assert a['config']['learning_rate'] == 0.01, a['config']['learning_rate']
assert b['config']['learning_rate'] == 0.1,  b['config']['learning_rate']
# nothing but the learning rate may differ
for k in ('epochs', 'delta_T', 'sparsity_scheduler', 'recovery_epochs',
          'val_split', 'prune_task_head', 'optimizer_type', 'num_classes',
          'dataset_name', 'batch_size'):
    assert a['config'][k] == b['config'][k], (k, a['config'][k], b['config'][k])
print(a['key'], '\n', b['key'])
assert nb.sanity_check([a, b]), 'sanity check failed'
nb.run_group([a, b], gpu=GPU)

## Verdict

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and 'cifar100' in k:
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

d  = acc.get(f'static.dense.{MODEL}.{DATASET}.dense.seed{SEED}')
p1 = acc.get(f'static.prune.{MODEL}.{DATASET}.s0.95.magnitude.seed{SEED}')
p2 = acc.get(f'static.prune.{MODEL}.{DATASET}.s0.95.magnitude.seed{SEED}.lr0.1')

f = lambda v: f'{v:.2f}' if v is not None else '  --  '
print(f'dense            {f(d)}')
print(f'I.P. @ lr 0.01   {f(p1)}   <- protocol')
print(f'I.P. @ lr 0.1    {f(p2)}')
if p1 is not None and p2 is not None:
    gap = p2 - p1
    print(f'\ndelta (0.1 - 0.01) = {gap:+.2f}')
    if gap > 1.0:
        print('0.1 WINS CLEARLY -- the CIFAR-10 split does NOT transfer.')
        print('Re-plan the grid before running it; do not report the 0.01 arm.')
    elif gap < -1.0:
        print('0.01 wins clearly -- protocol transfers. Run the grid as specified.')
    else:
        print('Within ~1 point -- no evidence the split fails to transfer.')
        print('Run the grid as specified, and say the probe was run.')